# 01 data

Step 1 of the guide: load Sparkov, strip PII, sort by card and time, split by time, basic EDA. Each stage below prints what the data looks like after it, so the transformation can be read off the notebook.

In [1]:
import os, sys, glob, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
for f in glob.glob("notebooks/*.ipynb"):          # new notebooks from the repo appear in Drive
    dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok")

Mounted at /content/drive
/content/transaction-fm
tfm ok


## How train / val / test are separated, and why they sit in one file

```
time  ---- 2019-01-01 ------------------ 2020-05-22 ------ 2020-06-21 ------ 2020-12-31 ---->
              train (fit everything here)   val (pick ckpt,    test (fraudTest.csv,
                                            thresholds)        scored once at the end)
```

- The split is by time and written into every row as a `split` column. `test_start` is the first timestamp of fraudTest.csv, so test is exactly the Kaggle test file; `val` is the 30 days before it; both boundaries are saved in `01_data.json`.
- One parquet, not three files: a test-period transaction must see the same card's earlier history (which is in the train period), the same way a production scorer would. Test looking back is not leakage.
- The forbidden direction is blocked in code: every fit-style function in steps 2 to 5 (bin edges, vocabulary, merchant top-N, XGBoost) starts with `assert_train_only(df)` and raises if a val or test row is present; history windows only look backward; checkpoints and thresholds are chosen on val; test is evaluated once in step 6.
- Tests: `tests/test_data.py::test_assert_train_only_blocks_val_and_test_rows` and `test_train_rows_all_precede_val_and_test`.

## Stage 0: raw files as downloaded from Kaggle

In [2]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from tfm.data import load_raw, clean, assign_splits, eda_stats, save_outputs, PII_COLUMNS

raw, test_start = load_raw(f"{ROOT}/data/raw")
print("rows:", len(raw), " columns:", len(raw.columns), " test_start (first row of fraudTest.csv):", test_start)
print("columns:", list(raw.columns))
raw.head(5)

rows: 1852394  columns: 23  test_start (first row of fraudTest.csv): 2020-06-21 12:14:25
columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


## Stage 1a: why each raw column is dropped, kept, or derived

Three rules decide a column's fate, and the table below shows the evidence for each, computed on the raw data:

1. **Does the tokenizer need it?** time, category, merchant, amount, a location signal, and the label are kept (step 2 of the guide; Wells Fargo's geo lesson).
2. **Is it constant on a card?** `max_distinct_values_on_one_card == 1` means every row of a card carries the same value. For a sequence model that is zero information inside the sequence, and it acts as a card identity label the model could memorise instead of learning behaviour. Dropped.
3. **Is it an id or a file artifact?** `trans_num` is unique per row (pure noise); `Unnamed: 0` is the csv row index. Dropped.

`cc_num` is also constant per card but is needed as the grouping key, so it becomes a dense `card_id` that never enters the model.


In [3]:
per_card_max = raw.groupby("cc_num").nunique().max()        # most distinct values any single card has, per column
per_card_max["cc_num"] = 1                                  # the group key itself: one value per card by definition
per_card_max = per_card_max.astype(int)
fate = {}
for c in raw.columns:
    if c in PII_COLUMNS:
        fate[c] = "drop: rule 2, constant per card" if per_card_max[c] == 1 else "drop: rule 3, id per row"
    elif c == "Unnamed: 0":
        fate[c] = "drop: rule 3, csv row index"
    elif c == "cc_num":
        fate[c] = "-> card_id (group key only, never a model input)"
    elif c in ("lat", "long", "merch_lat", "merch_long"):
        fate[c] = "-> dist_km (rule 1, location signal)"
    elif c == "trans_date_trans_time":
        fate[c] = "-> ts (rule 1)"
    else:
        fate[c] = "keep (rule 1)"
print("trans_num unique on every row:", raw["trans_num"].is_unique)
print("columns constant on every card:", sorted(per_card_max[per_card_max == 1].index.tolist()))
pd.DataFrame({"max_distinct_values_on_one_card": per_card_max, "fate": pd.Series(fate)})


trans_num unique on every row: True
columns constant on every card: ['cc_num', 'city', 'city_pop', 'dob', 'first', 'gender', 'job', 'last', 'lat', 'long', 'state', 'street', 'zip']


,max_distinct_values_on_one_card,fate
Unnamed: 0,4390,"drop: rule 3, csv row index"
amt,3808,keep (rule 1)
category,14,keep (rule 1)
cc_num,1,"-> card_id (group key only, never a model input)"
city,1,"drop: rule 2, constant per card"
city_pop,1,"drop: rule 2, constant per card"
dob,1,"drop: rule 2, constant per card"
first,1,"drop: rule 2, constant per card"
gender,1,"drop: rule 2, constant per card"
is_fraud,2,keep (rule 1)


## Stage 1b: where `dist_km` comes from

Each raw row carries two coordinate pairs. `lat, long` is the cardholder's home address (constant on a card). `merch_lat, merch_long` is where this transaction's merchant is; the Sparkov generator scatters it around the home, so it differs on every row. `dist_km` is the great-circle distance between the two, computed with the haversine formula (`_haversine_km` in `tfm.data`: degrees to radians, `a = sin²(Δlat/2) + cos(lat1)·cos(lat2)·sin²(Δlon/2)`, distance `= 2·6371·asin(√a)`). Only the distance enters the model; the raw coordinates are dropped by rule 2.

Three rows of one card, with the distance worked out, then the distance distribution for normal and fraud rows.


In [4]:
from tfm.data import _haversine_km

ex = raw[raw["cc_num"] == raw["cc_num"].iloc[0]].head(3)[["cc_num", "trans_date_trans_time", "lat", "long", "merch_lat", "merch_long", "is_fraud"]].copy()
ex["dist_km"] = _haversine_km(ex["lat"].values, ex["long"].values, ex["merch_lat"].values, ex["merch_long"].values)
display(ex)

d = pd.DataFrame({"dist_km": _haversine_km(raw["lat"].values, raw["long"].values, raw["merch_lat"].values, raw["merch_long"].values),
                  "is_fraud": raw["is_fraud"]})
print("dist_km by label (if the two rows look alike, distance carries little signal on this synthetic data):")
d.groupby("is_fraud")["dist_km"].describe(percentiles=[.05, .5, .95]).round(1)


,cc_num,trans_date_trans_time,lat,long,merch_lat,merch_long,is_fraud,dist_km
0,2703186189652095,2019-01-01 00:00:18,36.0788,-81.1781,36.011293,-82.048315,0,78.597568
1565,2703186189652095,2019-01-01 16:53:34,36.0788,-81.1781,35.083586,-80.625951,0,121.405859
2299,2703186189652095,2019-01-01 23:02:37,36.0788,-81.1781,35.532409,-82.170017,0,108.131953


dist_km by label (if the two rows look alike, distance carries little signal on this synthetic data):


,count,mean,std,min,5%,50%,95%,max
is_fraud,,,,,,,,
0,1842743.0,76.1,29.1,0.0,24.8,78.2,120.5,152.1
1,9651.0,76.3,28.9,0.7,25.5,78.1,120.3,144.5


## Stage 1: clean

Drops the 11 PII columns and the index column, replaces `cc_num` with a dense `card_id`, adds `dist_km` (cardholder to merchant, haversine), sorts by card then time.

In [5]:
df = clean(raw)
print("dropped:", PII_COLUMNS + ["Unnamed: 0", "cc_num", "lat", "long", "merch_lat", "merch_long"])
print("kept:   ", list(df.columns))
print("rows:", len(df), " cards:", df["card_id"].nunique())
df.head(8)

dropped: ['first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'city_pop', 'job', 'dob', 'trans_num', 'Unnamed: 0', 'cc_num', 'lat', 'long', 'merch_lat', 'merch_long']
kept:    ['card_id', 'ts', 'unix_time', 'category', 'merchant', 'amt', 'dist_km', 'is_fraud']
rows: 1852394  cards: 999


,card_id,ts,unix_time,category,merchant,amt,dist_km,is_fraud
0,0,2019-01-01 12:47:15,1325422035,misc_net,"fraud_Jones, Sawayn and Romaguera",7.27,127.606239,0
1,0,2019-01-02 08:44:57,1325493897,gas_transport,fraud_Berge LLC,52.94,110.308921,0
2,0,2019-01-02 08:47:36,1325494056,gas_transport,fraud_Luettgen PLC,82.08,21.787261,0
3,0,2019-01-02 12:38:14,1325507894,kids_pets,fraud_Daugherty LLC,34.79,87.204215,0
4,0,2019-01-02 13:10:46,1325509846,home,fraud_Beier and Sons,27.18,74.212965,0
5,0,2019-01-03 13:56:35,1325598995,shopping_net,fraud_Stamm-Witting,6.87,69.984956,0
6,0,2019-01-03 17:05:10,1325610310,food_dining,fraud_Conroy-Emard,8.43,26.099210,0
7,0,2019-01-04 13:59:55,1325685595,home,fraud_Pollich LLC,117.11,59.375952,0


Same card, consecutive rows, so you can see the time order and the gaps:

In [6]:
df[df["card_id"] == 0].head(10)

,card_id,ts,unix_time,category,merchant,amt,dist_km,is_fraud
0,0,2019-01-01 12:47:15,1325422035,misc_net,"fraud_Jones, Sawayn and Romaguera",7.27,127.606239,0
1,0,2019-01-02 08:44:57,1325493897,gas_transport,fraud_Berge LLC,52.94,110.308921,0
2,0,2019-01-02 08:47:36,1325494056,gas_transport,fraud_Luettgen PLC,82.08,21.787261,0
3,0,2019-01-02 12:38:14,1325507894,kids_pets,fraud_Daugherty LLC,34.79,87.204215,0
4,0,2019-01-02 13:10:46,1325509846,home,fraud_Beier and Sons,27.18,74.212965,0
5,0,2019-01-03 13:56:35,1325598995,shopping_net,fraud_Stamm-Witting,6.87,69.984956,0
6,0,2019-01-03 17:05:10,1325610310,food_dining,fraud_Conroy-Emard,8.43,26.099210,0
7,0,2019-01-04 13:59:55,1325685595,home,fraud_Pollich LLC,117.11,59.375952,0
8,0,2019-01-04 21:17:22,1325711842,personal_care,fraud_Monahan-Morar,26.74,66.302255,0
9,0,2019-01-05 00:42:24,1325724144,grocery_pos,fraud_Vandervort-Funk,105.20,81.700505,0


Quality checks after clean, printed so they can be read off the page (the same checks live in `tests/test_data.py`):

In [7]:
print("rows before / after clean:", len(raw), "/", len(df), " (nothing dropped:", len(raw) == len(df), ")")
print("nulls per column:", df.isna().sum().to_dict())
print("card_id count == distinct cc_num count:", df["card_id"].nunique() == raw["cc_num"].nunique(), f"({df['card_id'].nunique()} cards)")
mono = df.groupby("card_id")["ts"].apply(lambda s: s.is_monotonic_increasing).all()
print("ts non-decreasing within every card:", bool(mono))
assert len(raw) == len(df) and df.isna().sum().sum() == 0 and mono
print("stage 1 checks ok")


rows before / after clean: 1852394 / 1852394  (nothing dropped: True )
nulls per column: {'card_id': 0, 'ts': 0, 'unix_time': 0, 'category': 0, 'merchant': 0, 'amt': 0, 'dist_km': 0, 'is_fraud': 0}
card_id count == distinct cc_num count: True (999 cards)
ts non-decreasing within every card: True
stage 1 checks ok


## Stage 2: split by time

`test` = rows at or after the first fraudTest.csv timestamp; `val` = the 30 days before it; `train` = everything earlier. No rows dropped, no shuffling.

In [8]:
df = assign_splits(df, test_start=test_start, val_days=30)
val_start = test_start - pd.Timedelta(days=30)
print("train <", val_start, " <= val <", test_start, " <= test")
summary = df.groupby("split").agg(rows=("ts", "size"), first_ts=("ts", "min"), last_ts=("ts", "max"), fraud_rate=("is_fraud", "mean")).loc[["train", "val", "test"]]
summary

train < 2020-05-22 12:14:25  <= val < 2020-06-21 12:14:25  <= test


,rows,first_ts,last_ts,fraud_rate
split,,,,
train,1212704,2019-01-01 00:00:18,2020-05-22 12:13:32,0.005786
val,83971,2020-05-22 12:14:39,2020-06-21 12:13:37,0.005823
test,555719,2020-06-21 12:14:25,2020-12-31 23:59:34,0.003860


The rows right at the two boundaries, to show the labels switch exactly there:

In [ ]:
around = df.sort_values("ts")
b1 = around[around["ts"] >= val_start].head(2).index.tolist() + around[around["ts"] < val_start].tail(2).index.tolist()
b2 = around[around["ts"] >= test_start].head(2).index.tolist() + around[around["ts"] < test_start].tail(2).index.tolist()
df.loc[sorted(b1 + b2)].sort_values("ts")[["card_id", "ts", "amt", "is_fraud", "split"]]

## Stage 3a: one row per card

The guide asks for transactions per card, gap distribution, fraud over time, and whether fraud comes in bursts. `eda_stats` collapses these into percentiles for the json; this table is the per-card frame behind those percentiles. Then two numbers on how concentrated fraud is across cards.


In [ ]:
g = df.groupby("card_id")
cards = pd.DataFrame({
    "n_tx": g.size(),
    "first_ts": g["ts"].min(),
    "last_ts": g["ts"].max(),
    "n_fraud": g["is_fraud"].sum(),
    "median_gap_h": g["ts"].diff().dt.total_seconds().div(3600).groupby(df["card_id"]).median(),
})
cards["fraud_rate"] = cards["n_fraud"] / cards["n_tx"]
print("cards:", len(cards), " cards with any fraud:", int((cards["n_fraud"] > 0).sum()),
      " fraud rows on those cards: median", cards.loc[cards["n_fraud"] > 0, "n_fraud"].median(),
      " max", cards["n_fraud"].max())
display(cards.describe(percentiles=[.05, .25, .5, .75, .95]).round(2))
cards.sort_values("n_fraud", ascending=False).head(10)


## Stage 3b: what one fraud burst looks like

The card with the most fraud rows: all its fraud rows plus the three normal transactions before and after each run, in time order. `fraud_burst_share` in the json is the share of fraud rows whose previous fraud on the same card was within 24 h; this is one of those runs on the page.


In [ ]:
top = cards["n_fraud"].idxmax()
one = df[df["card_id"] == top].reset_index(drop=True)
one["gap_h"] = one["ts"].diff().dt.total_seconds().div(3600).round(2)
idx = one.index[one["is_fraud"] == 1]
keep = sorted(set(j for k in idx for j in range(max(0, k - 3), min(len(one), k + 4))))
print(f"card {top}: {len(one)} transactions, {len(idx)} fraud, fraud span {one.loc[idx, 'ts'].min()} to {one.loc[idx, 'ts'].max()}")
pd.set_option("display.max_rows", 200)
one.loc[keep, ["ts", "gap_h", "category", "merchant", "amt", "dist_km", "is_fraud", "split"]]


## Stage 3: EDA numbers the report needs

In [ ]:
import json
stats = eda_stats(df)
print(json.dumps({k: v for k, v in stats.items() if k != "fraud_rate_by_month"}, indent=2))

## Stage 4: save

`ROOT/data/processed/transactions.parquet` (one file, `split` column) and `ROOT/results/01_data.json` (stats plus the two boundary timestamps).

In [ ]:
stats = save_outputs(df, stats, out_dir=f"{ROOT}/data/processed", results_dir=f"{ROOT}/results", test_start=test_start, val_days=30)
print("val_start:", stats["val_start"], " test_start:", stats["test_start"])
back = pd.read_parquet(f"{ROOT}/data/processed/transactions.parquet")
print("parquet rows:", len(back), " columns:", list(back.columns))

## Charts: fraud rate by month, and the gap between a card's consecutive transactions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m = pd.Series(stats["fraud_rate_by_month"])
m.plot(ax=ax[0], marker="o")
ax[0].axvline(list(m.index).index(stats["val_start"][:7]), color="grey", ls="--", label="val start")
ax[0].axvline(list(m.index).index(stats["test_start"][:7]), color="red", ls="--", label="test start")
ax[0].set_title("fraud rate by month"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=45)

gaps = df.groupby("card_id")["ts"].diff().dt.total_seconds().div(3600).dropna()
ax[1].hist(gaps.clip(upper=240), bins=60)
ax[1].set_title("hours between consecutive transactions of a card (clipped at 240)")
plt.tight_layout()
plt.savefig(f"{ROOT}/results/01_data_eda.png", dpi=120)
plt.show()

## What to read off the numbers

- `fraud_rate` around 0.5%, and `fraud_rate` per split tells you whether test is harder or easier than train.
- `fraud_burst_share`: share of fraud rows whose previous fraud on the same card was within 24 h. High means frauds come in bursts, so card-level discovery and leakage checks matter.
- `gap_hours` percentiles decide the time-gap bins in step 2.
- `n_merchants` (about 700) decides the merchant top-N cutoff in step 2.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
    cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
    return cells(a) == cells(b)
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
    repo = f"notebooks/{os.path.basename(f)}"
    if not os.path.exists(repo) or same_code(f, repo):
        shutil.copy(f, repo)
    else:
        print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
!git log --oneline | head -1

In [ ]:
drive.flush_and_unmount()